In [1]:
#install specialized tools for PDF processing, high-speed vector search (faiss), and memory optimization
!pip -q install -U transformers accelerate bitsandbytes sentencepiece sentence-transformers faiss-cpu pypdf pandas==2.2.2

import os #for managing file paths and directories
import re #for cleaning text patterns (Regular Expressions)
import gc #for clearing unused data from the computer's memory
import glob #for finding all files that match a pattern (like *.pdf)
import time #for adding pauses to the workflow
import faiss #high-speed library for searching through millions of text fragments
import torch #the core engine for running neural network calculations
import numpy as np
import pandas as pd

from pypdf import PdfReader #tool to extract raw text from PDF documents
from sentence_transformers import SentenceTransformer #tool to convert text into mathematical coordinates (embeddings)
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig #core tools for the local LLM

#connect to google drive
from google.colab import drive
drive.mount('/content/drive')

#setup
#define which models to use, Qwen for speaking and E5 for understanding and searching
MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
EMBEDDING_MODEL = "intfloat/multilingual-e5-base"

TEST_FILE = "/content/drive/MyDrive/data/dataset_clean.csv"
RESULTS_FILE = "/content/drive/MyDrive/data/rag_results.csv"
TEMP_RESULTS_FILE = "/content/drive/MyDrive/data/rag_results_temp.csv"

#settings for how to cut long laws into smaller pieces (chunks)
CHUNK_SIZE = 800     #each law fragment will be roughly 800 characters long
CHUNK_OVERLAP = 100  #pieces will overlap by 100 characters to ensure context isn't lost
TOP_K = 2            #how many law fragments to show the AI for each question
MAX_NEW_TOKENS = 80  #limit the AI's answer length to keep it concise

SYSTEM_PROMPT = """Du bist ein Experte für österreichisches Steuerrecht.
Antworte auf Deutsch, korrekt, direkt und kurz.
Nutze nur den bereitgestellten Kontext.
Antworte in 1–2 vollständig abgeschlossenen Sätzen.
Zitiere die relevanten gesetzlichen Bestimmungen möglichst genau."""

print("Setup done")
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))



# find uploaded pdf files
pdf_files = sorted(glob.glob("/content/drive/MyDrive/data/*.pdf"))

print("\nPDF files found:")
for pdf in pdf_files:
    print(" -", os.path.basename(pdf))

if len(pdf_files) == 0:
    raise ValueError("No PDF files found in /content. Upload your RIS PDFs first.")

if not os.path.exists(TEST_FILE):
    raise ValueError("dataset_clean.csv not found in /content.")



# helper functions
# function to clean PDF text (like page numbers or website footers)
def clean_text(text):
    if text is None:
        return ""

    text = text.replace("\x00", " ") #remove null bytes that break the code
    text = text.replace("\u00ad", "")
    text = text.replace("￾", "")
    text = text.replace("­", "")
    text = text.replace("\r", "\n")

    lines = text.split("\n") #filter out standard RIS website headers and footers using line-by-line checks
    cleaned_lines = []

    for line in lines:
        line = line.strip()

        #remove obvious headers/footers from RIS print preview
        if line.startswith("RIS - "):
            continue
        if "https://www.ris.bka.gv.at/" in line:
            continue
        if "Seite " in line and "von" in line:
            continue
        if "Сторінка" in line and "із" in line:
            continue
        if re.match(r"^\d{2}\.\d{2}\.\d{2},\s*\d{2}:\d{2}$", line):
            continue

        cleaned_lines.append(line)

    text = "\n".join(cleaned_lines)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]{2,}", " ", text)
    text = text.strip()

    return text

#function to cut a long law into small, digestible chunks for the AI
def chunk_text(text, chunk_size=1200, overlap=200):
    if not text:
        return []

    text = re.sub(r"\n{3,}", "\n\n", text).strip()

    #split the text by paragraphs to preserve meaning
    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
    chunks = []
    current = ""

    for para in paragraphs: #group paragraphs until they reach the target size
        if len(current) + len(para) + 2 <= chunk_size:
            current = current + ("\n\n" if current else "") + para
        else:
            if current:
                chunks.append(current.strip())

            if len(para) <= chunk_size:
                current = para
            else:
                start = 0
                while start < len(para):
                    end = start + chunk_size
                    piece = para[start:end].strip()
                    if piece:
                        chunks.append(piece)
                    start += max(1, chunk_size - overlap)
                current = ""

    if current:
        chunks.append(current.strip())

    return chunks

#main function to open each PDF page and extract law text
def extract_pdf_chunks(pdf_path):
    print(f"\nReading PDF: {os.path.basename(pdf_path)}")

    reader = PdfReader(pdf_path) #open the PDF
    docs = []

    for page_num, page in enumerate(reader.pages, start=1):
        try:
            text = page.extract_text(extraction_mode="layout")
        except:
            try:
                text = page.extract_text()
            except Exception as e:
                print(f"  Error on page {page_num}: {e}")
                text = ""

        text = clean_text(text) #pull text and clean it

        if len(text) < 50: #skip empty or nearly empty pages
            continue

        page_chunks = chunk_text(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP)

        for i, ch in enumerate(page_chunks): #save the text fragment along with its metadata (source and page)
            docs.append({
                "source": os.path.basename(pdf_path),
                "page": page_num,
                "chunk_id": f"{os.path.basename(pdf_path)}_p{page_num}_c{i}",
                "text": ch
            })

        if page_num % 20 == 0:
            print(f"  Processed page {page_num}/{len(reader.pages)}")

    print(f"  Total chunks from file: {len(docs)}")
    return docs #save chunks with their source filename and page number


# extract all law texts into chunks
all_docs = []

for pdf_path in pdf_files:
    file_docs = extract_pdf_chunks(pdf_path)
    all_docs.extend(file_docs)

if len(all_docs) == 0:
    raise ValueError("No chunks were extracted from the PDFs.")

chunks_df = pd.DataFrame(all_docs) #store all law pieces in a table
print("\nTotal chunks in corpus:", len(chunks_df))
print(chunks_df.head(3))


# load embedding model
print("\nLoading embedding model...")

embedder = SentenceTransformer(EMBEDDING_MODEL) #load the embedding model (converts text into numbers/vectors)
print("Embedding model loaded")


# build embeddings + FAISS index
print("\nEncoding chunks...")

passage_texts = ["passage: " + t for t in chunks_df["text"].tolist()]
passage_embeddings = embedder.encode( #transform all law chunks into mathematical coordinates (embeddings)
    passage_texts,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True
).astype("float32")

print("Embeddings shape:", passage_embeddings.shape)

#create a FAISS index: a super-fast "map" to find which law fragment is closest to a question
index = faiss.IndexFlatIP(passage_embeddings.shape[1])
index.add(passage_embeddings)

print("FAISS index built")
print("Indexed vectors:", index.ntotal)


#retrieval function
#function to find the most relevant law paragraphs for a specific question
def retrieve_context(question, top_k=4):
    query_text = "query: " + question

    query_embedding = embedder.encode( #turn question into coordinates
        [query_text],
        normalize_embeddings=True,
        convert_to_numpy=True
    ).astype("float32")

    scores, indices = index.search(query_embedding, top_k) #find the 4 closest law chunks
    indices = indices[0]
    scores = scores[0]

    retrieved = []
    seen = set()

    for idx, score in zip(indices, scores):
        row = chunks_df.iloc[int(idx)]
        key = (row["source"], row["page"], row["text"][:120])

        if key in seen:
            continue
        seen.add(key)

        retrieved.append({
            "score": float(score),
            "source": row["source"],
            "page": int(row["page"]),
            "text": row["text"]
        })

    context_parts = []
    for i, item in enumerate(retrieved, start=1):
        context_parts.append(
            f"[Quelle {i}: {item['source']}, Seite {item['page']}]\n{item['text']}"
        )

    context = "\n\n".join(context_parts)
    return context, retrieved #return a single block of text containing these law fragments


#load local LLM in 4-bit
print("\nLoading RAG generation model...")

compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype
)

tokenizer = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

model.eval() #set to "evaluation mode" (inference only, no training)
print("RAG generation model loaded")


# generation function
def get_answer(question):
  # 1. search the laws for relevant context
    try:
        context, retrieved = retrieve_context(question, top_k=TOP_K)
  # 2. Build a prompt that gives the AI the law text and the question
        user_prompt = f"""Frage: {question}

Gesetzeskontext:
{context}

Beantworte die Frage nur auf Grundlage dieses Kontexts und nenne die passende Gesetzesstelle."""

        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt}
        ]
    #format the text for the Qwen model and send it to the GPU
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = tokenizer(
          text,
          return_tensors="pt",
          truncation=True,
          max_length=2048
        )
        inputs = {k: v.to(model.device) for k, v in inputs.items()}

    # 3. generate the answer using the local model
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                use_cache=True,
                pad_token_id=tokenizer.pad_token_id
            )

        answer = tokenizer.decode(
            outputs[0][inputs["input_ids"].shape[1]:],
            skip_special_tokens=True
        ).strip()

        if answer == "":
            return "ERROR", retrieved

        return answer, retrieved

    except Exception as e:
        print(f"Error: {e}")
        print("  Waiting 10 seconds...")
        time.sleep(10)
        return "ERROR", []


# load test dataset
df = pd.read_csv(TEST_FILE, encoding="utf-8-sig")
print(f"\nTotal questions loaded: {len(df)}")
print(df.head(3))


# main loop
answers = []
ids = []

for i, row in df.iterrows(): #loop through all 643 questions in your dataset clean file
    question_id = row["id"]
    question = row["prompt"]

    print(f"\n[{i+1}/{len(df)}] Processing: {question_id}")

    answer, retrieved = get_answer(question)

    answers.append(answer)
    ids.append(question_id)

    print(f"Preview: {answer[:120]}...")

    if len(retrieved) > 0:
      print(f"Top source: {retrieved[0]['source']} | page {retrieved[0]['page']}")

    time.sleep(0.5)

    if (i + 1) % 50 == 0:
        temp_df = pd.DataFrame({"id": ids, "answer": answers})
        temp_df.to_csv(TEMP_RESULTS_FILE, index=False)
        print(f"\nProgress saved: {i+1} questions done\n")



# save final output
final_df = pd.DataFrame({
    "id": ids,
    "answer": answers
})

final_df.to_csv(RESULTS_FILE, index=False)

print("\nDONE!")
print(f"Total answers: {len(final_df)}")
print(final_df.head(3))
print("\nSaved file:", RESULTS_FILE)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 52.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 47.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 18.9 MB/s eta 0:00:00
Mounted at /content/drive
Setup done
CUDA available: True
GPU: Tesla T4

PDF files found:
 - LLMs — RIS - Einkommensteuergesetz 1988 - Bundesrecht konsolidiert, Fassung vom 07.04.2026.pdf
 - RIS - Grunderwerbsteuergesetz 1987 - Bundesrecht konsolidiert, Fassung vom 07.04.2026.pdf
 - RIS - Körperschaftsteuergesetz 1988 - Bundesrecht konsolidiert, Fassung vom 07.04.2026.pdf
 - RIS - Umsatzsteuergesetz 1994 - Bundesrecht konsolidiert, Fassung vom 07.04.2026.pdf

Reading PDF: LLMs — RIS - Einkommensteuergesetz 1988 - Bundesrecht konsolidiert, Fassung vom 07.04.2026.pdf


  Processed page 20/265
  Processed page 40/265
  Processed page 60/265
  Processed page 80/265
  Processed page 100/265
  Processed page 120/265
  Processed page 140/265
  Processed page 160/265
  Processed page 180/265
  Processed page 200/265
  Processed page 220/265
  Processed page 240/265
  Processed page 260/265
  Total chunks from file: 1573

Reading PDF: RIS - Grunderwerbsteuergesetz 1987 - Bundesrecht konsolidiert, Fassung vom 07.04.2026.pdf


  Total chunks from file: 111

Reading PDF: RIS - Körperschaftsteuergesetz 1988 - Bundesrecht konsolidiert, Fassung vom 07.04.2026.pdf
  Processed page 20/60
  Processed page 40/60
  Processed page 60/60
  Total chunks from file: 356

Reading PDF: RIS - Umsatzsteuergesetz 1994 - Bundesrecht konsolidiert, Fassung vom 07.04.2026.pdf
  Processed page 20/83
  Processed page 40/83
  Processed page 60/83
  Processed page 80/83
  Total chunks from file: 497

Total chunks in corpus: 2537
                                              source  page  \
0  LLMs — RIS - Einkommensteuergesetz 1988 - Bund...     1   
1  LLMs — RIS - Einkommensteuergesetz 1988 - Bund...     1   
2  LLMs — RIS - Einkommensteuergesetz 1988 - Bund...     1   

                                            chunk_id  \
0  LLMs — RIS - Einkommensteuergesetz 1988 - Bund...   
1  LLMs — RIS - Einkommensteuergesetz 1988 - Bund...   
2  LLMs — RIS - Einkommensteuergesetz 1988 - Bund...   

                                        

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

RAG generation model loaded


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



Total questions loaded: 643
             id                                             prompt
0  CORP-TAX-001  Was ist die steuerliche Bemessungsgrundlage fü...
1  CORP-TAX-002  Welche steuerlichen Konsequenzen hat es, wenn ...
2  CORP-TAX-003  Welche Körperschaften sind verpflichtet, sämtl...

[1/643] Processing: CORP-TAX-001
Preview: Die Steuer ist zu berechnen vom Wert der Gegenleistung (§ 5), mindestens vom Wert des Grundstückes (§ 6). Abweichend von...
Top source: RIS - Grunderwerbsteuergesetz 1987 - Bundesrecht konsolidiert, Fassung vom 07.04.2026.pdf | page 7

[2/643] Processing: CORP-TAX-002
Preview: Es gibt keine steuerlichen Konsequenzen für das Verbrechen eines zinslosen Darlehens an einen nicht-zinslosen Gesellscha...
Top source: RIS - Körperschaftsteuergesetz 1988 - Bundesrecht konsolidiert, Fassung vom 07.04.2026.pdf | page 25

[3/643] Processing: CORP-TAX-003
Preview: In Österreich sind die Einkünfte von Gewerbebetrieb und Gewinnanteile von Gesellschaften und Kommandi